# **Analysis of embeddings: practice**

Hi, everyone! In this homework you will get to know the basic approaches for analysing embeddings and probing a model. We will work with the BERT model and analyse how different parts of speech are arranged inside it, and we will also look at whether the extracted activations are enough to solve the task of classifying semantic entities. As an additional task we will look at whether the words responsible for emotions group separately in the BERT model.

**To do this, you will:**

- Extract embeddings from the pretrained BERT model.
- Carry out dimensionality reduction in order to assess the visual representation of the different parts of speech and of the emotion cluster;
- Train a linear classifier in order to check whether the model encodes different semantic entities differently enough.

Enjoy the practice!

### **Block 1. Preparing the data and the model.**

**Step 1.**
As always, let us load the libraries.

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import seaborn as sns

import pandas as pd

Let us also prepare the dataset. It has been collected in advance and uploaded to GitHub.

In [ ]:
words_data = pd.read_csv('https://github.com/SadSabrina/explainable_AI_course/raw/refs/heads/main/HW_module14_LLM/words_data.csv', index_col=[0])
words_data.head()

**Step 2.**

Let us load the pretrained BERT model — *Bidirectional Encoder Representations* — one of the well-known pioneer models of the transformer architecture. Specifically, bert-base-uncased is a pretrained BERT model from Google. The training was carried out on a corpus of English data using the generation of a masked token (masked language modeling). The uncased model does not distinguish between upper- and lower-case letters of English text.

![bert_image](https://cdn-images-1.medium.com/v2/resize:fit:1500/1*g1KBCVCITjrd9IJ7AyFqdw.png)

In [ ]:
# Loading the BERT model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

In [ ]:
# Switching the model into prediction mode
model.eval()

**Task 1. Analyse the architecture of the BERT model. What is the dimensionality of the embeddings obtained at the output?**

**Task 2. Analyse the architecture of the BERT model. What is the dimensionality of the attention weights?**

**Step 3.** Let us write a function for extracting the embeddings from the last hidden layer.

In [ ]:
def extract_embeddings(words, model, tokenizer):
    """
    Extracts embeddings for a list of words using the BERT model.
    """
    inputs = tokenizer(words, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state
    # Returning the averaged embeddings
    return embeddings.median(dim=1).values

Let us go through the code in detail. First the input words are encoded as tokens. For the BERT model each word may be encoded by more than one token.

In [ ]:
tokenizer('cat') # For example, cat is encoded by three tokens, where tokens 101, 102 are the start and end tokens of the text sequence

**Task 3. Find the word that is encoded by the largest number of tokens.**

In [ ]:
# Your code here

Next, formally we obtain the prediction of the model. However, the `BertModel` that we use in the lesson does not have a layer for a final task, so at the output we have exactly what is needed for the analysis — the hidden states for each token in the sequence.

In [ ]:
from transformers import BertForQuestionAnswering

# But if we do solve a task — we will solve a task. Let us look at the question-answering task

model_example = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')

# The question and the text
question = "Who are cats?"
text = "Cats are the best animal of the world"

# Step 1: Tokenization
encoding = tokenizer.encode_plus(question, text, add_special_tokens=True, return_tensors="pt")

# Getting input_ids and token_type_ids
input_ids = encoding["input_ids"]
token_type_ids = encoding["token_type_ids"]

# Prediction using the model
with torch.no_grad():
    start_scores, end_scores = model_example(input_ids, token_type_ids=token_type_ids).start_logits, model_example(input_ids, token_type_ids=token_type_ids).end_logits

#  Converting the indices into tokens
all_tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

start_index = torch.argmax(start_scores)
end_index = torch.argmax(end_scores)

# Extracting the answer
answer = ' '.join(all_tokens[start_index:end_index+1])

print("Answer:", answer)

Let us return to our task.

In [ ]:
# Tokenizing the words
inputs = tokenizer(list(words_data['word']), return_tensors='pt', padding=True, truncation=True)

# Extracting the embeddings
features = extract_embeddings(list(words_data['word']), model, tokenizer)

print(f"Embedding dimensionality: {features.shape}")

Note that in the function we added an averaging of the embeddings. This is necessary, since originally the size of the hidden states is `torch.Size([91, 6, 768])`.

Which corresponds to:
- 91 words
- 6 vectors of dimension 768 per word, where each vector encodes a separate token.

After the averaging we obtained exactly the representation vector of each word from the task. And for better informativeness, we took the median.

**Task 4. How is each word encoded if we feed in a sequence of words which, generally speaking, are encoded by different numbers of tokens?**

### **Block 2. Visualization.**

In [ ]:
def visualize_embeddings(features, labels, words, method='pca'):
    """
    Visualizes embeddings with support for several categories.
    """
    if method == 'pca':
        reducer = PCA(n_components=2, random_state=42)
        title = "Visualization of the embeddings with PCA"
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=42)
        title = "Visualization of the embeddings with t-SNE"
    else:
        raise ValueError("The method must be 'pca' or 'tsne'")

    reduced_embeddings = reducer.fit_transform(features)
    plt.figure(figsize=(10, 8))

    # Unique categories
    unique_labels = sorted(set(labels))
    colors = plt.cm.get_cmap('tab10', len(unique_labels))
    categories = {0: "Noun", 1: "Verb", 2: "Adjective", 3: "Adverb", 4 : "Emotions"}

    for label in unique_labels:
        indices = [i for i, l in enumerate(labels) if l == label]
        plt.scatter(
            [reduced_embeddings[i, 0] for i in indices],
            [reduced_embeddings[i, 1] for i in indices],
            label=categories.get(label, f"Category {label}")
        )

    for i, word in enumerate(words):
        plt.text(reduced_embeddings[i, 0], reduced_embeddings[i, 1], word, fontsize=9)

    plt.title(title)
    plt.legend(loc='best')
    plt.show()

And so, the first approach to the analysis is visualization based on dimensionality reduction. We will use the PCA and t-SNE methods in order to project the embeddings onto a plane. The code is already written above.

In [ ]:
visualize_embeddings(features, words_data['label'], words_data['word'], method='pca')

**Task 5. Analyse the resulting clusters. Which categories (two of them) separated from the rest the most strongly?**

**Task 6. Analyse the clusters. Did the situation coincide with the reduction in PCA?**

In [ ]:
visualize_embeddings(features, words_data['label'], words_data['word'], method='tsne')

As you can see, different methods and strategies of dimensionality reduction will give a different picture. That is why the absence of separability does not mean the absence of separation in the model, although it does point to it.

### **Block 3. Probing.**

And so, at this stage we have seen that semantic entities group into separate clusters when projected onto a two-dimensional space. Is that enough to build a linear classifier based on the learned embeddings?

Let us investigate this question with the help of probing.  

In [ ]:
features.shape

**Please note:** the number of features is now larger than the number of observations. Therefore, the data has to be compressed in order to avoid overfitting. That is exactly what we will do in the first step.

Let us compress the data so that the components retain at least 70% of the explained variance of the data.

In [ ]:
pca = PCA(n_components=22)
features_pca = pca.fit_transform(features)

sum(pca.explained_variance_ratio_)

Let us try to solve two tasks.

1. Classification of parts of speech — nouns, adjectives, verbs and adverbs;
2. Classification of emotions — the emotion class against the rest;

Since emotions are expressed by nouns, in task 1 we will simply merge them with the nouns.

Class labels:
* 0 - noun;
* 1 - verb;
* 2 - adjective;
* 3 - adverb;
* 4 — emotions;
    

In [ ]:
features_pca_data = pd.DataFrame(features_pca, columns=[f'component_{i}' for i in range(1, 23)])

first_task_labels = list(map(lambda x: 0 if x == 4 else x, words_data['label']))
second_task_labels = list(map(lambda x: 0 if x != 4 else 1,  words_data['label']))

Let us train the models for the tasks and assess the quality.

In [ ]:
from sklearn.model_selection import train_test_split

features_pca_data_train, features_pca_data_test, first_task_labels_train, first_task_labels_test = train_test_split(features_pca_data,
                                                                                                                    first_task_labels,
                                                                                                                    random_state=42,
                                                                                                                    test_size=0.3)

second_task_labels_train, second_task_labels_test = train_test_split(second_task_labels, random_state=42, test_size=0.3)


In [ ]:
# Model initialization
first_task_model = LogisticRegression()
second_task_model = LogisticRegression()

Training and evaluation.

In [ ]:
# The first task

first_task_model.fit(features_pca_data_train, first_task_labels_train)
first_task_predictions = first_task_model.predict(features_pca_data_test)
accuracy = accuracy_score(first_task_labels_test, first_task_predictions)

print(f"First probing task Accuracy: {accuracy:.2f}")
print("\nFirst probing task classification Report:\n", classification_report(first_task_labels_test, first_task_predictions))

conf_matrix = confusion_matrix(first_task_labels_test, first_task_predictions)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Noun', 'Verb', 'Adj', 'Adv'], yticklabels=['Noun', 'Verb', 'Adj', 'Adv'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

**Task 7. Complete the code that trains the probe for the second task. Assess the accuracy of the model — what is the number of incorrect answers on the test data?**

In [ ]:
# The second task

second_task_model.fit(features_pca_data_train, second_task_labels_train)

second_task_predictions = # Your code here
accuracy = # Your code here

print(f"Second probing task Accuracy: {accuracy:.2f}")
print("\nSecond probing task classification Report:\n", classification_report(second_task_labels_test, second_task_predictions))

conf_matrix = confusion_matrix(second_task_labels_test, second_task_predictions)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Not emotion', 'Emotion'], yticklabels=['Not emotion', 'Emotion'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

## **Conclusion**

So, probing has shown that on the basis of the embeddings the data in the posed tasks can be separated linearly. Such results are not rare, and the linearity hypothesis — one of the central hypotheses of MI — is connected with them.

**Mechanistic interpretability (MI)** is a subfield of XAI which seeks to understand the internal mechanisms of how neural networks work. The linearity hypothesis, formulated by researchers in the field, assumes that the features 𝑓𝑖 in networks are linear combinations of neuron activations. The existence of linear probes shows that the concepts represented in the activations of a network can be extracted with the help of linear models.

But there are tasks in which a network forms nonlinear representations. One example is solving modular arithmetic tasks (for instance, working with dates), where the features cannot be separated by a linear hyperplane. This demonstrates that a linear representation is not a universal rule. But what we see empirically is amazing!

And the beauty of networks does not end there!
I hope the last task of the course made you feel even more interest in models.
Thank you for completing it!